# Statistical Inference for the Simple Linear Regression Model — Lecture Notebook
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW
**Based on:** Brooks, C. — *Introductory Econometrics for Finance*, Cambridge University Press, Ch. 3

---
**Learning Objectives:**
- Interpret $R^2$ as the share of variation in $y$ explained by the regression — and compute it from the variance decomposition $TSS = ESS + RSS$
- Compute the standard error $\widehat{\mathrm{SE}}(\hat{\beta}_1)$ by hand and understand what it measures
- Recognise *why* the test statistic follows Student's $t_{n-2}$ rather than $N(0,1)$ — and when the difference matters
- Carry out the four-step hypothesis test on a real CAPM regression and reach the same conclusion via the $t$-statistic, the $p$-value, and the 95% confidence interval
- Replace the classical OLS standard error with heteroskedasticity-consistent ($\mathrm{HC1}$) standard errors and judge when this matters

> Run each cell with **Shift+Enter**. This notebook accompanies the V5 lecture slides.


## Step 0 — Install & Import Libraries

In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Part 1 — The Running Example: Apple's CAPM Regression

Throughout this notebook we use the same regression as the lecture slides — Apple's daily returns on the S&P 500:

$$r_{AAPL,t} = \beta_0 + \beta_1 \, r_{SP500,t} + u_t$$

The slope $\hat{\beta}_1$ is Apple's **CAPM beta** — how much Apple moves for every 1% market move. The intercept $\hat{\beta}_0$ is the average daily excess (we are running a market model — see footnote at the end).


In [ ]:
# Download Apple and S&P 500 daily closing prices
data = yf.download(['AAPL', '^GSPC'], start='2019-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close']
ret  = data.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={'^GSPC': 'SP500'})[['AAPL', 'SP500']]
print(f'Observations: {len(ret)} trading days')
print(f'Period:       {ret.index[0].date()}  to  {ret.index[-1].date()}')
print(f'\nFirst 5 days (returns):')
print(ret.head().round(4))

In [ ]:
# Fit the OLS regression — this is the foundation for everything below
X = sm.add_constant(ret['SP500'])
y = ret['AAPL']
model = sm.OLS(y, X).fit()

# Extract the key quantities
beta0_hat = model.params['const']      # intercept
beta1_hat = model.params['SP500']      # slope (CAPM beta)
se_beta0  = model.bse['const']
se_beta1  = model.bse['SP500']
n_obs     = int(model.nobs)
df_resid  = int(model.df_resid)

print(f'β1_hat (slope)   = {beta1_hat:.4f}    ← Apple CAPM beta')
print(f'β0_hat (intercept) = {beta0_hat:.6f}')
print(f'n  = {n_obs}        df = n − 2 = {df_resid}')

---
# Part 2 — $R^2$: How Much Variation Does the Model Explain?

$R^2$ measures the share of the variation in $y$ that the regression line accounts for. It ranges from $0$ (the model explains nothing) to $1$ (the model explains everything perfectly).

**The variance decomposition** is the foundation:

$$\underbrace{\sum_{t=1}^{n}(y_t - \bar{y})^2}_{TSS \;=\; \text{total variation}} \;=\; \underbrace{\sum_{t=1}^{n}(\hat{Y}_t - \bar{y})^2}_{ESS \;=\; \text{explained by model}} \;+\; \underbrace{\sum_{t=1}^{n}\hat{u}_t^{\,2}}_{RSS \;=\; \text{residual / unexplained}}$$

Then

$$R^2 \;=\; \frac{ESS}{TSS} \;=\; 1 - \frac{RSS}{TSS}.$$


In [ ]:
# Compute R² by hand from the three sums of squares
y_bar  = y.mean()
yhat   = model.fittedvalues
uhat   = model.resid

TSS = ((y - y_bar) ** 2).sum()
ESS = ((yhat - y_bar) ** 2).sum()
RSS = (uhat ** 2).sum()

print(f'TSS = Σ(y − y_bar)²   = {TSS:.6f}')
print(f'ESS = Σ(Y_hat − y_bar)² = {ESS:.6f}')
print(f'RSS = Σ u_hat²       = {RSS:.6f}')
print(f'\nTSS  ≈  ESS + RSS ?   {np.isclose(TSS, ESS + RSS)}    ← variance decomposition holds')
print(f'\nR² = ESS / TSS = {ESS/TSS:.4f}')
print(f'R² = 1 − RSS/TSS = {1 - RSS/TSS:.4f}')
print(f'\nstatsmodels says:  R² = {model.rsquared:.4f}    ← identical')

### 2.1 Economic interpretation for Apple

The $R^2$ printed in Part 2 tells us what share **of the variation in Apple's daily returns is explained by movements in the S&P 500**. The remainder is **idiosyncratic risk** — Apple-specific news such as earnings releases, product launches, or lawsuits.

This matches the CAPM decomposition exactly:

$$\underbrace{\mathrm{Var}(r_{AAPL})}_{\text{total risk}} \;=\; \underbrace{\beta_1^{\,2} \cdot \mathrm{Var}(r_{SP500})}_{\text{systematic risk}} \;+\; \underbrace{\mathrm{Var}(u)}_{\text{firm-specific risk}}$$


In [ ]:
# Visualise the decomposition
fig, ax = plt.subplots(figsize=(9, 4.5))
shares = np.array([ESS/TSS, RSS/TSS]) * 100
colors = [YELLOW, ORANGE]
labels = [f'Explained (ESS / TSS = {shares[0]:.1f}%)\n  Systematic risk (market β)',
          f'Unexplained (RSS / TSS = {shares[1]:.1f}%)\n  Idiosyncratic risk (firm-specific)']

ax.barh([0], shares[0], color=colors[0], edgecolor='black', linewidth=0.6, label=labels[0])
ax.barh([0], shares[1], left=shares[0], color=colors[1], edgecolor='black', linewidth=0.6,
        label=labels[1])
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_xlabel('% of total variance in Apple returns')
ax.set_title(f"Variance decomposition for Apple's CAPM regression  (R² = {model.rsquared:.3f})",
             fontweight='bold', loc='left')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.4), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

---
# Part 3 — The Standard Error of $\hat{\beta}_1$: How Precise Is Our Estimate?

The slope $\hat{\beta}_1$ is a random variable. If we drew a different sample of trading days, we would get a different number. The **standard error** measures how much $\hat{\beta}_1$ would typically vary across hypothetical samples.

**The two formulas:**

$$\widehat{\mathrm{SE}}(\hat{\beta}_1) \;=\; \frac{\hat{\sigma}}{\sqrt{\sum_{t=1}^{n}(x_t - \bar{x})^2}}, \qquad \hat{\sigma}^2 \;=\; \frac{1}{n-2}\sum_{t=1}^{n}\hat{u}_t^{\,2} \;=\; \frac{RSS}{n-2}$$

- $\hat{\sigma}^2$ is the estimated residual variance.
- We divide $RSS$ by $n-2$ (not $n$) because two degrees of freedom were used estimating $\hat{\beta}_0$ and $\hat{\beta}_1$. This correction makes $\hat{\sigma}^2$ an unbiased estimator of the true error variance $\sigma^2$.


In [ ]:
# Step-by-step hand calculation of SE(β1_hat) for Apple
# ---------------------------------------------------

# Step 1: residual variance sigma²_hat
sigma2_hat = RSS / df_resid
print(f'Step 1: sigma²_hat = RSS / (n−2) = {RSS:.6f} / {df_resid} = {sigma2_hat:.6e}')

# Step 2: sum of squared x-deviations (denominator of SE formula)
x_bar = ret['SP500'].mean()
Sxx   = ((ret['SP500'] - x_bar) ** 2).sum()
print(f'Step 2: Σ(x − x_bar)² = {Sxx:.6f}')

# Step 3: plug into SE formula
se_beta1_manual = np.sqrt(sigma2_hat / Sxx)
print(f'Step 3: SE(β1_hat) = √(sigma²_hat / Σ(x − x_bar)²) = √({sigma2_hat:.6e} / {Sxx:.4f})')
print(f'                  = {se_beta1_manual:.4f}')

# Verify against statsmodels
print(f'\nstatsmodels:    SE(β1_hat) = {se_beta1:.4f}')
print(f'Identical? {np.isclose(se_beta1_manual, se_beta1)}    ← ✓')

### 3.1 What does the standard error tell us?

If we repeated the data-generating process — drawing another sample of daily returns of the same length — and ran OLS each time, the slope estimates would form a roughly bell-shaped distribution centred near the true $\beta_1$. The standard error tells us the **width** of that distribution.

Concretely: with the $\hat{\beta}_1$ and $\widehat{\mathrm{SE}}(\hat{\beta}_1)$ printed above, the estimate would typically deviate from its centre by about $\pm\widehat{\mathrm{SE}}$ across hypothetical samples.

A simulation makes this concrete.


In [ ]:
# Bootstrap-style simulation: resample with replacement and re-estimate β1_hat
np.random.seed(42)
B = 5000  # number of bootstrap replications
betas = np.empty(B)
n     = len(ret)
sp_vals = ret['SP500'].values
ap_vals = ret['AAPL'].values

for b in range(B):
    idx = np.random.randint(0, n, n)
    xb, yb = sp_vals[idx], ap_vals[idx]
    Sxx_b = ((xb - xb.mean()) ** 2).sum()
    Sxy_b = ((xb - xb.mean()) * (yb - yb.mean())).sum()
    betas[b] = Sxy_b / Sxx_b

print(f'Bootstrap mean of β1_hat: {betas.mean():.4f}    (≈ original {beta1_hat:.4f})')
print(f'Bootstrap std  of β1_hat: {betas.std():.4f}    (≈ formula SE {se_beta1:.4f})')
print('\n→ The bootstrap reproduces the sampling variability of β1_hat. Note that a PAIRS')
print('  bootstrap is heteroskedasticity-robust: it converges to the ROBUST SE of Part 9,')
print('  not to the classical formula. Under homoskedasticity the two coincide.')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(betas, bins=50, color=YELLOW, edgecolor='black', linewidth=0.4, alpha=0.85)
ax.axvline(beta1_hat, color=RED, lw=2, label=f'β1_hat (original) = {beta1_hat:.4f}')
ax.axvline(beta1_hat - se_beta1, color=GREY, ls='--', lw=1.2, label=f'±1 SE = ±{se_beta1:.4f}')
ax.axvline(beta1_hat + se_beta1, color=GREY, ls='--', lw=1.2)
ax.set_xlabel(r'β1_hat across bootstrap samples')
ax.set_ylabel('Frequency')
ax.set_title(f'Bootstrap sampling distribution of β1_hat — {B:,} resamples',
             fontweight='bold', loc='left')
ax.legend(loc='upper right', frameon=False)
plt.tight_layout(); plt.show()

---
# Part 4 — Why Student's $t$ and Not the Normal?

If we knew the true error variance $\sigma^2$, the standardised slope would follow a standard normal:

$$z \;=\; \frac{\hat{\beta}_1 - \beta_{1,H_0}}{\sigma \,/\, \sqrt{\sum (x_t - \bar{x})^2}} \;\sim\; N(0, 1)$$

But we never know $\sigma$. We *estimate* it with $\hat{\sigma}$, which is itself a random variable. This adds an extra source of uncertainty in the denominator. The corrected ratio follows **Student's $t$ with $n - 2$ degrees of freedom**:

$$t \;=\; \frac{\hat{\beta}_1 - \beta_{1,H_0}}{\widehat{\mathrm{SE}}(\hat{\beta}_1)} \;\sim\; t_{n-2}$$

For small $n$, the $t$-distribution has noticeably fatter tails than $N(0,1)$ — so critical values are larger. As $n \to \infty$, the two distributions become identical.


In [ ]:
# Plot t-densities at several degrees of freedom against N(0,1)
xx = np.linspace(-4.5, 4.5, 600)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
ax.plot(xx, stats.norm.pdf(xx),     color='black', lw=2.5,
        label='N(0, 1) — standard normal')
ax.plot(xx, stats.t.pdf(xx, 2),     color=RED,    lw=1.8, label=r'$t_{2}$  (very small df)')
ax.plot(xx, stats.t.pdf(xx, 5),     color=ORANGE, lw=1.8, label=r'$t_{5}$  (small df)')
ax.plot(xx, stats.t.pdf(xx, 30),    color=BLUE,   lw=1.8, ls='--',
        label=r'$t_{30}$ (large df ≈ N(0,1))')
ax.set_xlabel('t-value'); ax.set_ylabel('density')
ax.set_title('A.  Densities — t approaches N(0,1) as df grows',
             fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False, fontsize=9.5)

# Right panel — critical-value table
ax = axes[1]
ax.axis('off')
dfs = [5, 10, 20, 30, 50, 100, 1000, np.inf]
rows = []
for df_ in dfs:
    if np.isinf(df_):
        tc_10 = stats.norm.ppf(0.95)
        tc_05 = stats.norm.ppf(0.975)
        tc_01 = stats.norm.ppf(0.995)
        df_lbl = '∞'
    else:
        tc_10 = stats.t.ppf(0.95, df_)
        tc_05 = stats.t.ppf(0.975, df_)
        tc_01 = stats.t.ppf(0.995, df_)
        df_lbl = str(df_)
    rows.append([df_lbl, f'{tc_10:.3f}', f'{tc_05:.3f}', f'{tc_01:.3f}'])

cellColours = [[YELLOW]*4] + [['#F5F5F5' if i%2==0 else 'white']*4 for i in range(len(rows))]
the_tbl = ax.table(
    cellText = [['df = n−2', 'α = 0.10', 'α = 0.05', 'α = 0.01']] + rows,
    cellColours = cellColours,
    loc='center', cellLoc='center')
the_tbl.auto_set_font_size(False)
the_tbl.set_fontsize(10)
the_tbl.scale(1, 1.5)
ax.set_title(r'B.  Two-sided critical values  $t_{\alpha/2,\,df}$',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()
print('Rule of thumb: ALWAYS use t. It is correct in small samples and harmless in large ones.')

---
# Part 5 — The $t$-Statistic and $p$-Value: Testing Hypotheses

For the simple hypothesis test

$$H_0:\;\beta_1 = \beta_{1,H_0} \qquad \text{vs.} \qquad H_1:\;\beta_1 \neq \beta_{1,H_0}$$

the test statistic is

$$t \;=\; \frac{\hat{\beta}_1 - \beta_{1,H_0}}{\widehat{\mathrm{SE}}(\hat{\beta}_1)} \;\sim\; t_{n-2} \quad \text{under } H_0$$

and the **two-sided $p$-value** is

$$p\text{-value} \;=\; 2 \cdot \Pr\!\left( T_{n-2} > \lvert t_{\text{obs}} \rvert \right).$$

We reject $H_0$ whenever $\lvert t_{\text{obs}} \rvert > t_{\text{crit}}$  — equivalently, whenever the $p$-value falls below $\alpha$.


### 5.1 Example A — Testing whether Apple's slope differs from zero

$H_0: \beta_1 = 0$ vs. $H_1: \beta_1 \neq 0$.


In [ ]:
# Example A: Is Apple's slope statistically different from zero?
beta1_H0_A = 0
t_obs_A   = (beta1_hat - beta1_H0_A) / se_beta1
p_value_A = 2 * (1 - stats.t.cdf(abs(t_obs_A), df=df_resid))
t_crit    = stats.t.ppf(0.975, df=df_resid)

print('===  Example A: H0: β1 = 0  vs.  H1: β1 ≠ 0  ===\n')
print(f'Inputs:  β1_hat = {beta1_hat:.4f}    SE(β1_hat) = {se_beta1:.4f}    df = {df_resid}')
print(f'\nt-statistic:  t_obs = ({beta1_hat:.4f} − 0) / {se_beta1:.4f} = {t_obs_A:.2f}')
print(f'Critical value (α = 5%, two-sided):  t_crit = {t_crit:.3f}')
print(f'p-value:       p = 2 · P(T_{df_resid} > {abs(t_obs_A):.2f}) ≈ {p_value_A:.3e}')

print('\nDecision:')
if abs(t_obs_A) > t_crit:
    print(f'  |t_obs| = {abs(t_obs_A):.2f} > {t_crit:.3f} = t_crit   →   REJECT H0')
    print(f'  p-value ≈ {p_value_A:.1e} < 0.05           →   REJECT H0')
    print('  → Apple is clearly market-driven. The slope is highly significant.')
else:
    print(f'  |t_obs| = {abs(t_obs_A):.2f} < {t_crit:.3f} = t_crit   →   DO NOT REJECT H0')
    print(f'  p-value ≈ {p_value_A:.3f} > 0.05           →   DO NOT REJECT H0')

### 5.2 Example B — Testing whether the intercept differs from zero

$H_0: \beta_0 = 0$ vs. $H_1: \beta_0 \neq 0$.


In [ ]:
# Example B: Is Apple's intercept statistically different from zero?
beta0_H0_B = 0
t_obs_B   = (beta0_hat - beta0_H0_B) / se_beta0
p_value_B = 2 * (1 - stats.t.cdf(abs(t_obs_B), df=df_resid))

print('===  Example B: H0: β0 = 0  vs.  H1: β0 ≠ 0  ===\n')
print(f'Inputs:  β0_hat = {beta0_hat:.6f}    SE(β0_hat) = {se_beta0:.6f}    df = {df_resid}')
print(f'\nt-statistic:  t_obs = {beta0_hat:.6f} / {se_beta0:.6f} = {t_obs_B:.2f}')
print(f'Critical value (α = 5%):  t_crit = {t_crit:.3f}')
print(f'p-value:       p = 2 · P(T_{df_resid} > {abs(t_obs_B):.2f}) ≈ {p_value_B:.3f}')

print('\nDecision:')
if abs(t_obs_B) < t_crit:
    print(f'  |t_obs| = {abs(t_obs_B):.2f} < {t_crit:.3f} = t_crit   →   DO NOT REJECT H0')
    print(f'  p-value ≈ {p_value_B:.2f} > 0.05            →   DO NOT REJECT H0')
    print('  → We cannot rule out a zero intercept.')
else:
    print(f'  |t_obs| = {abs(t_obs_B):.2f} > {t_crit:.3f} = t_crit   →   REJECT H0')
    print(f'  p-value ≈ {p_value_B:.3f} < 0.05            →   REJECT H0')
    print('  → The intercept differs significantly from zero in this sample.')

### 5.3 Verify against `model.summary()`

`statsmodels` prints all of this for us automatically — the t-statistics, the p-values, and the confidence intervals — in the regression output.


In [ ]:
print(model.summary())

---
# Part 6 — Confidence Intervals: A Range, Not a Point

A 95% confidence interval is a range that, in repeated sampling, would contain the true $\beta_1$ in 95% of cases:

$$\hat{\beta}_1 \;\pm\; t_{\alpha/2,\, n-2} \cdot \widehat{\mathrm{SE}}(\hat{\beta}_1)$$

**Connection to the $t$-test:** if $0$ lies inside the 95% CI, we cannot reject $H_0:\beta_1 = 0$ at 5%. If $0$ lies outside, we reject. The CI and the $t$-test give exactly the same accept/reject decision — they are the same procedure expressed two different ways.


In [ ]:
# Build the 95% CI for slope and intercept by hand
ci_lo_b1 = beta1_hat - t_crit * se_beta1
ci_hi_b1 = beta1_hat + t_crit * se_beta1

ci_lo_b0 = beta0_hat - t_crit * se_beta0
ci_hi_b0 = beta0_hat + t_crit * se_beta0

print('95% Confidence Intervals (hand-built):')
print(f'  β1_hat:  [{ci_lo_b1:.4f},  {ci_hi_b1:.4f}]')
print(f'  β0_hat:  [{ci_lo_b0:.6f},  {ci_hi_b0:.6f}]')

print('\nstatsmodels:')
print(model.conf_int(alpha=0.05).round(6))

# Decision via the CI
print('\nDecisions implied by the CI:')
if not (ci_lo_b1 <= 0 <= ci_hi_b1):
    print(f'  0 is OUTSIDE the slope CI    → reject H0: β1 = 0 ✓')
if not (ci_lo_b1 <= 1 <= ci_hi_b1):
    # Rejecting H0: β1 = 1 says the beta differs from 1. WHICH WAY it differs
    # is read off the sign of (β1_hat − 1), not off the rejection itself.
    side = 'MORE' if beta1_hat > 1 else 'LESS'
    print(f'  1 is OUTSIDE the slope CI    → reject H0: β1 = 1 ✓')
    print(f'       β1_hat = {beta1_hat:.4f} < 1, so the direction is: {side} volatile than the market'
          if beta1_hat < 1 else
          f'       β1_hat = {beta1_hat:.4f} > 1, so the direction is: {side} volatile than the market')
else:
    print(f'  1 is INSIDE the slope CI     → cannot reject H0: β1 = 1')
if ci_lo_b0 <= 0 <= ci_hi_b0:
    print(f'  0 is INSIDE the intercept CI → cannot reject H0: β0 = 0 ✓')

In [ ]:
# Visualise the two CIs as a forest plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4),
                         gridspec_kw=dict(width_ratios=[1, 1.2]))

# Panel A — intercept
ax = axes[0]
ax.errorbar([beta0_hat], [0],
            xerr=[[beta0_hat - ci_lo_b0], [ci_hi_b0 - beta0_hat]],
            fmt='o', color=RED, ecolor='black', capsize=10, lw=2.5, ms=12)
ax.axvline(0, color=GREY, ls='--', lw=1.2)
ax.text(0, 0.5, r'$\beta_0 = 0$', ha='center', fontsize=10, color=GREY)
ax.text(beta0_hat, 0.3, fr'$\hat{{\beta}}_0 = {beta0_hat:.4f}$',
        ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(-0.6, 0.75); ax.set_yticks([])
ax.set_xlabel(r'$\hat{\beta}_0$  (intercept)')
ax.set_title(f'A.  95% CI for $\\hat{{\\beta}}_0$:  p = {p_value_B:.2f}  →  '
             f'{"reject" if p_value_B < 0.05 else "cannot reject"}',
             fontweight='bold', loc='left')

# Panel B — slope
ax = axes[1]
ax.errorbar([beta1_hat], [0],
            xerr=[[beta1_hat - ci_lo_b1], [ci_hi_b1 - beta1_hat]],
            fmt='o', color=RED, ecolor='black', capsize=10, lw=2.5, ms=12)
ax.axvline(0, color=GREY, ls='--', lw=1.2)
ax.axvline(1, color=ORANGE, ls=':', lw=1.5)
ax.text(0, 0.5, r'$\beta_1 = 0$', ha='center', fontsize=10, color=GREY)
ax.text(1, 0.5, r'$\beta_1 = 1$', ha='center', fontsize=10, color=ORANGE)
ax.text(beta1_hat, 0.3, fr'$\hat{{\beta}}_1 = {beta1_hat:.4f}$',
        ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(-0.6, 0.75); ax.set_yticks([])
ax.set_xlim(-0.15, 1.55)
ax.set_xlabel(r'$\hat{\beta}_1$  (slope)')
ax.set_title(f'B.  95% CI for $\\hat{{\\beta}}_1$:  p = {model.pvalues["SP500"]:.1e}  →  '
             f'{"reject" if model.pvalues["SP500"] < 0.05 else "cannot reject"}',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

---
# Part 7 — The Four-Step Hypothesis Test: Is Apple Market-Neutral?

A more interesting question than "is $\beta_1 = 0$?" is "is $\beta_1 = 1$?" — i.e., does Apple move one-for-one with the market, or is it more (or less) volatile? Let's run the formal four-step test.

**Step 1 — State the hypotheses:**  
$H_0: \beta_1 = 1$  vs.  $H_1: \beta_1 \neq 1$

**Step 2 — Choose significance level $\alpha$ and critical value:**  
$\alpha = 5\%$, two-sided  →  $t_{\text{crit}} \approx \pm 1.96$ for the $\mathrm{df} = n - 2$ that your download produces (the exact value is printed in Part 2).

**Step 3 — Compute the test statistic:**  

$$t \;=\; \frac{\hat{\beta}_1 - 1}{\widehat{\mathrm{SE}}(\hat{\beta}_1)}$$

**Step 4 — Make the decision:** reject if $|t| > t_{\text{crit}}$.


In [ ]:
# Four-step test: H0: β1 = 1 (market-neutrality)
beta1_H0 = 1
t_obs = (beta1_hat - beta1_H0) / se_beta1
p_val = 2 * (1 - stats.t.cdf(abs(t_obs), df=df_resid))

print('=== Four-step hypothesis test: H0: β1 = 1 ===\n')
print(f'Step 1.  Hypotheses:        H0: β1 = 1  vs.  H1: β1 ≠ 1')
print(f'Step 2.  α = 5%, two-sided → t_crit = ±{t_crit:.3f}')
print(f'Step 3.  Test statistic:')
print(f'         t = (β1_hat − 1) / SE(β1_hat)')
print(f'           = ({beta1_hat:.4f} − 1) / {se_beta1:.4f}')
print(f'           = {beta1_hat - 1:.4f} / {se_beta1:.4f}')
print(f'           = {t_obs:.2f}')
print(f'         p-value = {p_val:.3e}')
print(f'\nStep 4.  Decision:')
if abs(t_obs) > t_crit:
    print(f'         |{t_obs:.2f}| > {t_crit:.3f}   →   REJECT H0')
    # A two-sided rejection only says "beta is not 1". The DIRECTION comes from
    # the sign of (beta1_hat - 1), so read it off the estimate, never off the test.
    side = 'MORE' if beta1_hat > 1 else 'LESS'
    print(f'         → beta1_hat = {beta1_hat:.4f}, i.e. {"above" if beta1_hat > 1 else "below"} 1,')
    print(f'           so Apple is statistically {side} volatile than the broad market.')
else:
    print(f'         |{t_obs:.2f}| < {t_crit:.3f}   →   DO NOT REJECT H0')
    print(f'         → beta1_hat = {beta1_hat:.4f} is not significantly different from 1:')
    print(f'           the data are consistent with Apple moving one-for-one with the market.')

In [ ]:
# statsmodels has this built in — t_test
print('Verify with statsmodels t_test:')
print(model.t_test('SP500 = 1'))

---
# Part 8 — Robust Standard Errors: When the Classical Assumptions Break

The classical SE formula assumes the residuals have constant variance — **homoskedasticity** (Brooks Ch. 3 assumption A2). In financial data this rarely holds. Volatility clusters: large moves are followed by large moves, regardless of $x$.

When residuals are **heteroskedastic**, the classical SE is biased — typically too small — leading us to over-reject $H_0$. The fix is the **heteroskedasticity-consistent (HC) standard error**. For the simple slope, the scalar form is:

$$\widehat{\mathrm{SE}}_{\mathrm{HC}}(\hat{\beta}_1) \;=\; \sqrt{\frac{\sum_{t=1}^{n}(x_t - \bar{x})^2 \, \hat{u}_t^{\,2}}{\left[\sum_{t=1}^{n}(x_t - \bar{x})^2\right]^2}}$$

Each squared $x$-deviation is **weighted by its own squared residual** $\hat{u}_t^{\,2}$ — observations with bigger residuals contribute more to the SE. The point estimate $\hat{\beta}_1$ is **unchanged** — only the standard error changes.

In Python: pass `cov_type='HC1'` to `.fit()`.


In [ ]:
# Compute the HC1 robust standard error and compare to OLS
model_hc = sm.OLS(y, X).fit(cov_type='HC1')

# Manual scalar HC formula (informational — slightly different from HC0/HC1 small-sample correction)
x_centered = ret['SP500'] - x_bar
se_hc_manual = np.sqrt(((x_centered ** 2 * uhat ** 2).sum()) / (Sxx ** 2))

print('Comparison: OLS vs. HC1 robust standard errors')
print('-' * 55)
print(f'                        OLS           HC1 (robust)')
print(f'  β1_hat                {beta1_hat:.4f}        {beta1_hat:.4f}   ← unchanged')
print(f'  SE(β1_hat)            {se_beta1:.4f}        {model_hc.bse["SP500"]:.4f}')
print(f'  t-statistic           {beta1_hat/se_beta1:.2f}         {beta1_hat/model_hc.bse["SP500"]:.2f}')
print(f'  p-value               {model.pvalues["SP500"]:.3e}     {model_hc.pvalues["SP500"]:.3e}')
print(f'  95% CI                [{ci_lo_b1:.4f}, {ci_hi_b1:.4f}]   '
      f'[{model_hc.conf_int().loc["SP500"][0]:.4f}, {model_hc.conf_int().loc["SP500"][1]:.4f}]')
print(f'\nManual scalar HC0 formula (no df correction): {se_hc_manual:.4f}')
_gap = (model_hc.bse['SP500'] / se_beta1 - 1) * 100
print(f'\n→ Here the robust SE differs from the classical one by {_gap:+.1f}% — for messier data')
print('  (strong heteroskedasticity, influential outliers) the gap can be far larger.')

### 8.1 Visualising heteroskedasticity in the Apple residuals

Plotting residuals against fitted values reveals whether the variance changes with the level of the fitted value (a typical signature of heteroskedasticity).


In [ ]:
# Residuals-vs-fitted and Breusch-Pagan test for heteroskedasticity
from statsmodels.stats.diagnostic import het_breuschpagan

bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(model.resid, model.model.exog)
print('Breusch-Pagan test for heteroskedasticity:')
print(f'  LM statistic    = {bp_lm:.2f}')
print(f'  LM p-value      = {bp_lm_p:.4e}')
print('  H0: homoskedasticity     vs.     H1: heteroskedasticity')
if bp_lm_p < 0.05:
    print('  → Reject H0: there IS evidence of heteroskedasticity → use HC1 standard errors.\n')
else:
    print('  → Do not reject H0: classical SE is fine.\n')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: residuals over time
ax = axes[0]
ax.plot(model.resid.index, model.resid.values, color=GREY, lw=0.6)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Date'); ax.set_ylabel('Residual u_hat')
ax.set_title('A.  Residuals over time — look for variance clusters',
             fontweight='bold', loc='left')

# Right: residuals vs fitted
ax = axes[1]
ax.scatter(model.fittedvalues, model.resid, s=8, alpha=0.35, color=GREY)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Fitted value Y_hat'); ax.set_ylabel('Residual u_hat')
ax.set_title('B.  Residuals vs fitted — look for fanning shape',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

---
## Summary Table

| Concept | Key Formula | Python (after `model = sm.OLS(y, X).fit()`) |
|---------|-------------|------------------|
| Coefficient of determination | $R^2 = ESS / TSS = 1 - RSS/TSS$ | `model.rsquared` |
| Residual variance | $\hat{\sigma}^2 = RSS / (n-2)$ | `model.mse_resid` |
| Standard error of slope | $\widehat{\mathrm{SE}}(\hat{\beta}_1) = \hat{\sigma}/\sqrt{\sum(x-\bar{x})^2}$ | `model.bse['SP500']` |
| $t$-statistic | $t = (\hat{\beta}_1 - \beta_{1,H_0}) / \widehat{\mathrm{SE}}(\hat{\beta}_1)$ | `model.tvalues['SP500']` |
| $p$-value | $p = 2 \cdot \Pr(T_{n-2} > \lvert t \rvert)$ | `model.pvalues['SP500']` |
| 95% confidence interval | $\hat{\beta}_1 \pm t_{0.025,\,n-2} \cdot \widehat{\mathrm{SE}}$ | `model.conf_int()` |
| Robust SE (HC1) | scalar weighted sandwich | `sm.OLS(y,X).fit(cov_type='HC1')` |
| Custom hypothesis test | n/a | `model.t_test('SP500 = 1')` |

**Visual diagnostic cheat sheet:**
- $\lvert t \rvert$ large + $p$-value tiny → reject $H_0$, the relationship is real
- 0 outside the 95% CI → equivalent rejection
- Breusch-Pagan $p$-value < 0.05 → use HC1 standard errors

**Note on this notebook's CAPM regression.** We regressed raw Apple returns on raw S&P 500 returns — the **market model**, not the true CAPM with excess returns. For daily data with $r_f \approx 0.01\%$, the difference is negligible for the slope (only the intercept's interpretation as Jensen's alpha would change). We follow this convention because $r_f$ adds noise without adding insight at the daily frequency.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*Next: Multiple Linear Regression — controlling for many factors, the F-test for joint significance, adjusted $R^2$.*
